In [ ]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import os
import re
import warnings
import joblib
import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

print("✅ All libraries imported successfully!")


In [4]:
import pandas as pd
import numpy as np
import os
import re
import joblib
import warnings

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

print("Imports successful!")

Imports successful!


In [6]:
# Load the dataset
dataset_path = '../data/youtube_data.csv'
df = pd.read_csv(dataset_path)

print(f"Dataset loaded with shape: {df.shape}")

# Display the first few rows of the dataset
print("\nFirst few rows of the dataset:")
print(df.head())

# Basic data exploration
print("\nDataset information:")
print(df.info()) 

Dataset loaded with shape: (17589, 17)

First few rows of the dataset:
      video_id  duration  bitrate  bitrate(video)  height  width  frame rate  \
0  --F7dc-_FSI       180     5777            5640    1920   1080       25.00   
1  --cCAD-8Y_U       930     1195            1001    1280    720       30.00   
2  --g2gG8pQ0w       233     3028            2833    1280    720       23.98   
3  -0DR7-voRCU       562      431             300     320    240       29.97   
4  -0Fkp-2EzX0       300     3087            2929    1280    720       23.98   

   frame rate(est.) codec         category  \
0             25.00  h264  News & Politics   
1             30.00  h264           Gaming   
2             23.98  h264           Comedy   
3              0.00  h264   People & Blogs   
4             23.98  h264           Gaming   

                                           url  \
0  https://www.youtube.com/watch?v=--F7dc-_FSI   
1  https://www.youtube.com/watch?v=--cCAD-8Y_U   
2  https://www.youtub

In [4]:
import pandas as pd

df = pd.read_csv(dataset_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (17589, 17)

Columns:
['video_id', 'duration', 'bitrate', 'bitrate(video)', 'height', 'width', 'frame rate', 'frame rate(est.)', 'codec', 'category', 'url', 'title', 'description', 'hashtags', 'views', 'likes', 'comments']


In [5]:
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
video_id               0
duration               0
bitrate                0
bitrate(video)         0
height                 0
width                  0
frame rate             0
frame rate(est.)       0
codec                  0
category               0
url                    0
title                  0
description         1112
hashtags            1576
views                  0
likes                  0
comments               0
dtype: int64


In [7]:
print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())


First 5 rows:


,video_id,duration,bitrate,bitrate(video),height,width,frame rate,frame rate(est.),codec,category,url,title,description,hashtags,views,likes,comments
0,--F7dc-_FSI,180,5777,5640,1920,1080,25.00,25.00,h264,News & Politics,https://www.youtube.com/watch?v=--F7dc-_FSI,«السودان ينتفض» أمام السفارة بالقاهرة,حرية سلام وعدالة، الثورة خيار الشعب.. بهذا اله...,"في, عسكري, حكم, حرية, بخصوص, المصري اليوم, الس...",4078,41,3
1,--cCAD-8Y_U,930,1195,1001,1280,720,30.00,30.00,h264,Gaming,https://www.youtube.com/watch?v=--cCAD-8Y_U,Pokemon Tower Defense Episodio 2,Espero que tenham gostado do video!,"Pokémon (Anime/Manga Franchise), Pokémon (Vide...",33,2,1
2,--g2gG8pQ0w,233,3028,2833,1280,720,23.98,23.98,h264,Comedy,https://www.youtube.com/watch?v=--g2gG8pQ0w,New Hip Hop - Kemo Treats - Pancakes,Download all our music for FREE just enter $0....,"breaking bad spoof, Recipes, Pancakes, canada,...",13227,135,17
3,-0DR7-voRCU,562,431,300,320,240,29.97,0.00,h264,People & Blogs,https://www.youtube.com/watch?v=-0DR7-voRCU,Sister's Keeper Impression,A Review of the Shampoo Bar(Sunburst) and Dip ...,"natural hair, first impression, sister's keepe...",43,1,0
4,-0Fkp-2EzX0,300,3087,2929,1280,720,23.98,23.98,h264,Gaming,https://www.youtube.com/watch?v=-0Fkp-2EzX0,JBHAMMER777 (AKUMA) VS KtossPol Rank 1 (Vega)...,JBHAMMER777 is he strong enough to fight again...,"Tokido, momochi, (Ryu), AE, 11, at, Piyoppia, ...",222,2,0



Missing values:
video_id               0
duration               0
bitrate                0
bitrate(video)         0
height                 0
width                  0
frame rate             0
frame rate(est.)       0
codec                  0
category               0
url                    0
title                  0
description         1112
hashtags            1576
views                  0
likes                  0
comments               0
dtype: int64


In [ ]:
print(df.columns.tolist())

In [8]:
print("Columns in dataset:")
print(df.columns.tolist())

print("\nDoes 'Trending' column exist?")
print('Trending' in df.columns)

Columns in dataset:
['video_id', 'duration', 'bitrate', 'bitrate(video)', 'height', 'width', 'frame rate', 'frame rate(est.)', 'codec', 'category', 'url', 'title', 'description', 'hashtags', 'views', 'likes', 'comments']

Does 'Trending' column exist?
False


In [7]:
# Create a binary Trending label from views
# Top 10% most-viewed videos = Trending (1)
# Remaining videos = Not Trending (0)

view_threshold = df['views'].quantile(0.90)

df['Trending'] = (df['views'] >= view_threshold).astype(int)

print("Trending column created!")
print("\nView threshold:", view_threshold)

print("\nDistribution:")
print(df['Trending'].value_counts())

print("\nPercentage:")
print(df['Trending'].value_counts(normalize=True) * 100)

Trending column created!

View threshold: 5580.000000000004

Distribution:
Trending
0    15830
1     1759
Name: count, dtype: int64

Percentage:
Trending
0    89.999431
1    10.000569
Name: proportion, dtype: float64


In [10]:
print(df[['title', 'views', 'Trending']].head(10)) 

                                               title   views  Trending
0              «السودان ينتفض» أمام السفارة بالقاهرة    4078         0
1                 Pokemon Tower Defense   Episodio 2      33         0
2               New Hip Hop - Kemo Treats - Pancakes   13227         1
3                         Sister's Keeper Impression      43         0
4  JBHAMMER777 (AKUMA) VS KtossPol  Rank 1 (Vega)...     222         0
5                                     D'Salapa mirae     168         0
6                                  VATOz N DA BARRIO     138         0
7                                    Shiiiiiiiiiiiit  373587         1
8  LAURA POTHIN . DANS MES VEINES BY SUNSHINE974PROD    1876         0
9                Boonaa Mohammed - Killing with Love      50         0


In [8]:
# Clean missing text values
df['title'] = df['title'].fillna('').astype(str)
df['description'] = df['description'].fillna('').astype(str)
df['hashtags'] = df['hashtags'].fillna('').astype(str)

# Create combined text
df['combined_text'] = (
    df['title'] + ' ' +
    df['description'] + ' ' +
    df['hashtags']
)

print("Text preprocessing completed!")
print("\nMissing values after cleaning:")
print(df[['title', 'description', 'hashtags', 'combined_text']].isnull().sum())

print("\nExample combined text:")
print(df['combined_text'].iloc[0][:500]) 

Text preprocessing completed!

Missing values after cleaning:
title            0
description      0
hashtags         0
combined_text    0
dtype: int64

Example combined text:
«السودان ينتفض» أمام السفارة بالقاهرة حرية سلام وعدالة، الثورة خيار الشعب.. بهذا الهتاف تجمع عدد من أبناء الجالية السودانية أما مقر سفارتهم بالقاهرة مطالبين برحيل البشير، وطالب المتظاهرون الرئيس المصري المنتخب بإعلان التضامن مع ثورة الشعب السوداني وقطع كافة العلاقات مع نظام البشير.  تقرير: هيثم ذكي مونتاج: إيناس مرزوق في, عسكري, حكم, حرية, بخصوص, المصري اليوم, السودان, قوات, from, مصر, مكافحة, ديني, حول, Video Journalism, مظاهرت, video, Journalism, in, عنف, ثورة, الأمن, البشير, تعذيب, صحافة, احت


In [9]:
# Create basic text features

df['title_length'] = df['title'].str.len()
df['description_length'] = df['description'].str.len()
df['description_word_count'] = df['description'].str.split().str.len()
df['tag_count'] = df['hashtags'].str.split().str.len()

df['has_description'] = (df['description'].str.strip() != '').astype(int)
df['has_tags'] = (df['hashtags'].str.strip() != '').astype(int)
df['title_has_question'] = df['title'].str.contains(r'\?', regex=True).astype(int)

print("Feature engineering completed!")

print("\nNew features:")
print(df[
    [
        'title_length',
        'description_length',
        'description_word_count',
        'tag_count',
        'has_description',
        'has_tags',
        'title_has_question'
    ]
].head()) 

Feature engineering completed!

New features:
   title_length  description_length  description_word_count  tag_count  \
0            37                 281                      44         50   
1            34                  35                       6         18   
2            36                1957                     290         75   
3            26                  57                       9         15   
4            76                 357                      55         55   

   has_description  has_tags  title_has_question  
0                1         1                   0  
1                1         1                   0  
2                1         1                   0  
3                1         1                   0  
4                1         1                   0  


In [10]:
from sklearn.model_selection import train_test_split

# Features and target
X = df[
    [
        'title',
        'description',
        'hashtags',
        'title_length',
        'description_length',
        'description_word_count',
        'tag_count',
        'has_description',
        'has_tags',
        'title_has_question'
    ]
]

y = df['Trending']

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data split completed!")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Data split completed!
Training samples: 14071
Testing samples: 3518

Training class distribution:
Trending
0    12664
1     1407
Name: count, dtype: int64

Testing class distribution:
Trending
0    3166
1     352
Name: count, dtype: int64


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TF-IDF vectorizers
tfidf_title = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

tfidf_description = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2)
)

tfidf_tags = TfidfVectorizer(
    max_features=3000,
    stop_words='english',
    ngram_range=(1, 2)
)

# Fit ONLY on training data
title_train = tfidf_title.fit_transform(X_train['title'])
title_test = tfidf_title.transform(X_test['title'])

description_train = tfidf_description.fit_transform(X_train['description'])
description_test = tfidf_description.transform(X_test['description'])

tags_train = tfidf_tags.fit_transform(X_train['hashtags'])
tags_test = tfidf_tags.transform(X_test['hashtags'])

print("TF-IDF conversion completed!")

print("Title features:", title_train.shape)
print("Description features:", description_train.shape)
print("Hashtag features:", tags_train.shape)

TF-IDF conversion completed!
Title features: (14071, 5000)
Description features: (14071, 5000)
Hashtag features: (14071, 3000)


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================================
# IMPROVED TF-IDF FEATURES
# ============================================================

tfidf_title = TfidfVectorizer(
    max_features=8000,
    stop_words='english',
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_description = TfidfVectorizer(
    max_features=8000,
    stop_words='english',
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_tags = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Fit ONLY on training data
title_train = tfidf_title.fit_transform(X_train['title'])
title_test = tfidf_title.transform(X_test['title'])

description_train = tfidf_description.fit_transform(X_train['description'])
description_test = tfidf_description.transform(X_test['description'])

tags_train = tfidf_tags.fit_transform(X_train['hashtags'])
tags_test = tfidf_tags.transform(X_test['hashtags'])

print("Improved TF-IDF conversion completed!")

print("Title features:", title_train.shape)
print("Description features:", description_train.shape)
print("Hashtag features:", tags_train.shape)

Improved TF-IDF conversion completed!
Title features: (14071, 8000)
Description features: (14071, 8000)
Hashtag features: (14071, 5000)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)
import time

print("Training Logistic Regression...")

start_time = time.time()

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

logistic_model.fit(X_train_final, y_train)

training_time = time.time() - start_time

# Predictions
y_pred_lr = logistic_model.predict(X_test_final)

# Evaluation
accuracy_lr = accuracy_score(y_test, y_pred_lr)
precision_lr = precision_score(y_test, y_pred_lr)
recall_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print("\nLogistic Regression Results")
print("---------------------------")
print(f"Training time: {training_time:.2f} seconds")
print(f"Accuracy:      {accuracy_lr:.4f}")
print(f"Precision:     {precision_lr:.4f}")
print(f"Recall:        {recall_lr:.4f}")
print(f"F1 Score:      {f1_lr:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

Training Logistic Regression...

Logistic Regression Results
---------------------------
Training time: 5.40 seconds
Accuracy:      0.8209
Precision:     0.2509
Recall:        0.3977
F1 Score:      0.3077

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.87      0.90      3166
           1       0.25      0.40      0.31       352

    accuracy                           0.82      3518
   macro avg       0.59      0.63      0.60      3518
weighted avg       0.86      0.82      0.84      3518



In [14]:

from scipy.sparse import hstack, csr_matrix

numeric_columns = [
    'title_length',
    'description_length',
    'description_word_count',
    'tag_count',
    'has_description',
    'has_tags',
    'title_has_question'
]

numeric_train = csr_matrix(X_train[numeric_columns].values)
numeric_test = csr_matrix(X_test[numeric_columns].values)

X_train_final = hstack([
    title_train,
    description_train,
    tags_train,
    numeric_train
]).tocsr()

X_test_final = hstack([
    title_test,
    description_test,
    tags_test,
    numeric_test
]).tocsr()

print("Final feature matrix created!")
print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

Final feature matrix created!
Training shape: (14071, 21007)
Testing shape: (3518, 21007)


In [13]:
from sklearn.ensemble import RandomForestClassifier

print("Training Random Forest...")

start_time = time.time()

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_final, y_train)

training_time_rf = time.time() - start_time

y_pred_rf = rf_model.predict(X_test_final)

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("\nRandom Forest Results")
print("--------------------")
print(f"Training time: {training_time_rf:.2f} seconds")
print(f"Accuracy:      {accuracy_rf:.4f}")
print(f"Precision:     {precision_rf:.4f}")
print(f"Recall:        {recall_rf:.4f}")
print(f"F1 Score:      {f1_rf:.4f}")

Training Random Forest...

Random Forest Results
--------------------
Training time: 4.83 seconds
Accuracy:      0.8587
Precision:     0.2474
Recall:        0.2017
F1 Score:      0.2222


In [18]:
print("Logistic Regression:", logistic_model is not None)
print("Random Forest:", rf_model is not None)
print("Training features:", X_train_final.shape[1])
print("Testing features:", X_test_final.shape[1])

Logistic Regression: True
Random Forest: True
Training features: 21007
Testing features: 21007


In [19]:
import os
import joblib

# Save models in the project's root models folder
os.makedirs('../models', exist_ok=True)

# Select Logistic Regression as the best model
best_model = logistic_model

# Save the model
joblib.dump(best_model, '../models/best_model.pkl')

# Save its matching text vectorizers
joblib.dump(tfidf_title, '../models/tfidf_title.pkl')
joblib.dump(tfidf_description, '../models/tfidf_description.pkl')
joblib.dump(tfidf_tags, '../models/tfidf_tags.pkl')

print("Model and TF-IDF vectorizers saved successfully!")

print("Saved files:")
print(os.listdir('../models'))

Model and TF-IDF vectorizers saved successfully!
Saved files:
['.ipynb_checkpoints', 'best_model.pkl', 'tfidf_description.pkl', 'tfidf_tags.pkl', 'tfidf_title.pkl']


In [19]:
print("MODEL COMPARISON")
print("=" * 50)

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy_lr, accuracy_rf],
    'Precision': [precision_lr, precision_rf],
    'Recall': [recall_lr, recall_rf],
    'F1 Score': [f1_lr, f1_rf]
})

display(results.round(4))

MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.8081,0.2365,0.4119,0.3005
1,Random Forest,0.8579,0.2628,0.2330,0.2470


In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob_lr = logistic_model.predict_proba(X_test_final)[:, 1]

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("Threshold Comparison")
print("=" * 60)

for threshold in thresholds:
    y_pred_threshold = (y_prob_lr >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold Comparison
Threshold: 0.20 | Precision: 0.1392 | Recall: 0.8295 | F1: 0.2385
Threshold: 0.25 | Precision: 0.1644 | Recall: 0.7585 | F1: 0.2702
Threshold: 0.30 | Precision: 0.1793 | Recall: 0.6790 | F1: 0.2837
Threshold: 0.35 | Precision: 0.1945 | Recall: 0.6080 | F1: 0.2948
Threshold: 0.40 | Precision: 0.2064 | Recall: 0.5142 | F1: 0.2945
Threshold: 0.45 | Precision: 0.2160 | Recall: 0.4517 | F1: 0.2923
Threshold: 0.50 | Precision: 0.2365 | Recall: 0.4119 | F1: 0.3005


In [20]:

import os
import joblib

os.makedirs('../models', exist_ok=True)

best_model = logistic_model

joblib.dump(best_model, '../models/best_model.pkl')
joblib.dump(tfidf_title, '../models/tfidf_title.pkl')
joblib.dump(tfidf_description, '../models/tfidf_description.pkl')
joblib.dump(tfidf_tags, '../models/tfidf_tags.pkl')

print("Model and TF-IDF vectorizers saved successfully!")

Model and TF-IDF vectorizers saved successfully!


In [21]:

import os

files = [
    'best_model.pkl',
    'tfidf_title.pkl',
    'tfidf_description.pkl',
    'tfidf_tags.pkl'
]

for file in files:
    path = os.path.join('../models', file)
    print(f"{file}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

best_model.pkl: FOUND
tfidf_title.pkl: FOUND
tfidf_description.pkl: FOUND
tfidf_tags.pkl: FOUND


In [22]:

import joblib
import os

os.makedirs('../models', exist_ok=True)

joblib.dump(X_test_final, '../models/X_test_final.pkl')
joblib.dump(y_test, '../models/y_test.pkl')

print("Test data saved successfully!")
print("X_test_final shape:", X_test_final.shape)
print("y_test shape:", y_test.shape)

Test data saved successfully!
X_test_final shape: (3518, 21007)
y_test shape: (3518,)


In [23]:

import os

files = [
    'best_model.pkl',
    'tfidf_title.pkl',
    'tfidf_description.pkl',
    'tfidf_tags.pkl',
    'X_test_final.pkl',
    'y_test.pkl'
]

for file in files:
    path = os.path.join('../models', file)
    print(f"{file}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

best_model.pkl: FOUND
tfidf_title.pkl: FOUND
tfidf_description.pkl: FOUND
tfidf_tags.pkl: FOUND
X_test_final.pkl: FOUND
y_test.pkl: FOUND
